# 🧪 Phase 4: Advanced Deep Learning Models (BERT & RoBERTa)

> **⚠️ COMPUTATION WARNING:**
> Fine-tuning Transformer models like BERT and RoBERTa requires immense computational power. If you are running this on a standard laptop without a dedicated NVIDIA GPU, this notebook will take hours or days to complete.
>
> **Recommendation:** Upload this notebook and your `data/processed/` folder to **Google Colab** and enable the T4 GPU runtime (`Runtime > Change runtime type > T4 GPU`) to train these models in minutes.

---

## 1. Environment Setup

In [1]:
# Run this cell if you are in Google Colab to install dependencies
!pip install -q transformers torch datasets scikit-learn pandas

In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

# Ensure GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## 2. Dataset Preparation
We will load our cleaned datasets. For demonstration purposes and fast iteration, we will sample the dataset. You can increase the sample size for final portfolio results.

In [6]:
# Load data (Update path if running on Google Colab Drive)
try:
    df_news = pd.read_csv('/content/isot_cleaned.csv', engine='python', on_bad_lines='warn')
    df_tweets = pd.read_csv('/content/covid_tweets_cleaned.csv')
except FileNotFoundError:
    # Fallback for Colab if uploaded directly
    print("Please ensure 'isot_cleaned.csv' and 'covid_tweets_cleaned.csv' are uploaded.")
    df_news = pd.read_csv('isot_cleaned.csv', engine='python', on_bad_lines='warn')
    df_tweets = pd.read_csv('covid_tweets_cleaned.csv')

# --- SAMPLE SIZES ---
# Change these to len(df) to train on everything
NEWS_SAMPLE = len(df_news)
TWEET_SAMPLE = len(df_tweets)

df_news = df_news.sample(n=NEWS_SAMPLE, random_state=42).reset_index(drop=True)
df_tweets = df_tweets.sample(n=TWEET_SAMPLE, random_state=42).reset_index(drop=True)

print(f"News dataset: {len(df_news)} samples")
print(f"Tweet dataset: {len(df_tweets)} samples")

News dataset: 812 samples
Tweet dataset: 10699 samples


/tmp/ipykernel_15108/3159230876.py:3: ParserWarning: Skipping line 814: unexpected end of data

  df_news = pd.read_csv('/content/isot_cleaned.csv', engine='python', on_bad_lines='warn')


---
## 3. Fine-Tuning BERT for News Articles
We use `bert-base-uncased` for the long-form news articles.

In [7]:
news_model_name = "bert-base-uncased"
news_tokenizer = AutoTokenizer.from_pretrained(news_model_name)

def tokenize_news(batch):
    return news_tokenizer(batch["text_clean"], padding="max_length", truncation=True, max_length=256)

# Prepare Dataset
news_ds = Dataset.from_pandas(df_news[['text_clean', 'label_binary']])
news_ds = news_ds.rename_column("label_binary", "labels")
news_ds = news_ds.map(tokenize_news, batched=True)

# Train/Test Split
news_split = news_ds.train_test_split(test_size=0.2, seed=42)
train_news_ds = news_split['train']
test_news_ds = news_split['test']

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/812 [00:00<?, ? examples/s]

In [8]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Load Model
news_model = AutoModelForSequenceClassification.from_pretrained(news_model_name, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir="./results_news",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10,
)

news_trainer = Trainer(
    model=news_model,
    args=training_args,
    train_dataset=train_news_ds,
    eval_dataset=test_news_ds,
    compute_metrics=compute_metrics,
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
news_trainer.train() # Uncomment this line to execute training!
print(news_trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.008742,0.004849,1.000000,1.000000,1.000000,1.000000
2,0.001968,0.001405,1.000000,1.000000,1.000000,1.000000
3,0.001380,0.001076,1.000000,1.000000,1.000000,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.0010755074908956885, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_runtime': 2.3941, 'eval_samples_per_second': 68.083, 'eval_steps_per_second': 4.595, 'epoch': 3.0}


---
## 4. Fine-Tuning RoBERTa for Tweets
We use `cardiffnlp/twitter-roberta-base`, a model pre-trained on 58M tweets. It natively understands hashtags, handles '@' mentions, and parses Twitter-specific semantics much better than standard BERT.

In [10]:
tweet_model_name = "cardiffnlp/twitter-roberta-base"
tweet_tokenizer = AutoTokenizer.from_pretrained(tweet_model_name)

def tokenize_tweets(batch):
    # Tweets are short, so max_length 128 is plenty
    return tweet_tokenizer(batch["text_clean"], padding="max_length", truncation=True, max_length=128)

# Prepare Dataset
tweet_ds = Dataset.from_pandas(df_tweets[['text_clean', 'label_binary']])
tweet_ds = tweet_ds.rename_column("label_binary", "labels")
tweet_ds = tweet_ds.map(tokenize_tweets, batched=True)

# Train/Test Split
tweet_split = tweet_ds.train_test_split(test_size=0.2, seed=42)
train_tweet_ds = tweet_split['train']
test_tweet_ds = tweet_split['test']

config.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/10699 [00:00<?, ? examples/s]

In [11]:
# Load Model
tweet_model = AutoModelForSequenceClassification.from_pretrained(tweet_model_name, num_labels=2).to(device)

training_args_tweet = TrainingArguments(
    output_dir="./results_tweets",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
)

tweet_trainer = Trainer(
    model=tweet_model,
    args=training_args_tweet,
    train_dataset=train_tweet_ds,
    eval_dataset=test_tweet_ds,
    compute_metrics=compute_metrics,
)

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
tweet_trainer.train() # Uncomment this line to execute training!
print(tweet_trainer.evaluate())

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.400017,0.359544,0.842991,0.809740,0.751050,0.878378
2,0.299277,0.413201,0.848598,0.816535,0.757353,0.885749
3,0.248673,0.427365,0.855140,0.824064,0.765823,0.891892


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.42736494541168213, 'eval_accuracy': 0.8551401869158879, 'eval_f1': 0.8240635641316686, 'eval_precision': 0.7658227848101266, 'eval_recall': 0.8918918918918919, 'eval_runtime': 16.9163, 'eval_samples_per_second': 126.505, 'eval_steps_per_second': 7.921, 'epoch': 3.0}
